In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import matplotlib.colors as mcolors
from scipy.optimize import fsolve
import os
import pandas as pd
from fenics import *
from mshr import *
import math
from reaktoro import *
import time

# Global Config: PATHS (local machine, Miniconda)

In [2]:
BASE_PATH = '/home/jesus/LSWF_Carmona_Programming/D.61057/'

Figure_Path        = os.path.join(BASE_PATH, 'Figures/')
Data_Path          = os.path.join(BASE_PATH, 'Data/')
Transport_Path     = os.path.join(BASE_PATH, 'Transport_Results/')
Transport_Figures  = os.path.join(Transport_Path, 'Figures/')

for p in [BASE_PATH, Figure_Path, Data_Path, Transport_Path, Transport_Figures]:
    os.makedirs(p, exist_ok=True)

print("Using base path:", BASE_PATH)
print("Data directory:", Data_Path)
print("Figure directory (main):", Figure_Path)
print("Transport results:", Transport_Path)
print("Transport figures:", Transport_Figures)

Using base path: /home/jesus/LSWF_Carmona_Programming/D.61057/
Data directory: /home/jesus/LSWF_Carmona_Programming/D.61057/Data/
Figure directory (main): /home/jesus/LSWF_Carmona_Programming/D.61057/Figures/
Transport results: /home/jesus/LSWF_Carmona_Programming/D.61057/Transport_Results/
Transport figures: /home/jesus/LSWF_Carmona_Programming/D.61057/Transport_Results/Figures/


# Global Configuration

In [3]:
data_exp = np.loadtxt(os.path.join(Data_Path, "recuperacion.txt"), delimiter=',')
xp_all = data_exp[:, 0]   # hours
yp_all = data_exp[:, 1]   # % OIIP

### Fenics Settings 

In [4]:
parameters["form_compiler"]["cpp_optimize"] = True
parameters["form_compiler"]["optimize"] = True
import dolfin
dolfin.set_log_level(dolfin.LogLevel.ERROR)

### Unit Convertion

In [5]:
millidarcy = 1e-15
milliliter = 1e-6
psi = 6894.757
minute = 60
hour = 3600
day = 24 * hour
cm = 0.01


### Core and Fluid Properties 

In [6]:
L = 0.4              # core length [m]
d = 0.05               # core diameter [m]
phi = 0.6              # porosity
A = np.pi * (d / 2) ** 2
mu_w = 0.0056             # water viscosity [Pa*s]
mu_o = 0.034             # oil viscosity [Pa*s]
q_total = 1e-6 / 3600     # 1 cm3/h -> m3/s
u_total = q_total / A     # total superficial velocity [m/s]


### Function Space, grid 

In [7]:
fs_order_transport = 1
PecletGlobal = 100
R = 1
D_xy = 0.
q_c = 0.
NY = 100
xmin, ymin = 0., 0.
xmax, ymax = d, L
radius = d / 2


### Brine Composition

In [8]:

HSWF = {
    'Na': 2, 'Ca': 1, 'Mg': 0.08,
    'Cl': 1.1, 'SO4': 1.4, 'HCO3': 1.3,
}
MSWF = {
    'Na': 1.8, 'Ca': 0.9, 'Mg': 0.05,
    'Cl': 1, 'SO4': 1.2, 'HCO3': 1.1,
}
LSWF = {
    'Na': 1.5, 'Ca': 0.5, 'Mg': 0.3,
    'Cl': 0.9, 'SO4': 1.1, 'HCO3': 0.9,
}

# Brooks-Corey parameters for the three stages (end-point values)
KR_HSWF = dict(krw_max=0.050, kro_max=0.015, swc=0.0867, sor=0.60, nw=2.0, no=1.5)
KR_MSWF = dict(krw_max=0.040, kro_max=0.060, swc=0.0867, sor=0.60, nw=2.5, no=2.0)
KR_LSWF = dict(krw_max=0.030, kro_max=0.075, swc=0.0867, sor=0.51, nw=2, no=2.0)


CONCENTRATION_STAGES = {
    'HSWF': {
        't_start': 0.0,
        't_end': 159.379368331 * hour,
        'C0': HSWF,
        'KR': KR_HSWF,
    },
    'MSWF': {
        't_start': 159.379368331 * hour,
        't_end': 381.360912196 * hour,
        'C0': MSWF,
        'KR': KR_MSWF,
    },
    'LSWF': {
        't_start': 381.360912196 * hour,
        't_end': 548.084647600 * hour,
        'C0': LSWF,
        'KR': KR_LSWF,
    },
}

STAGE_ORDER = sorted(CONCENTRATION_STAGES.keys(),
                      key=lambda k: CONCENTRATION_STAGES[k]['t_start'])
KR_PHASE_PARAMS_LIST = [CONCENTRATION_STAGES[name]['KR'] for name in STAGE_ORDER]


SWITCH_TIMES = [CONCENTRATION_STAGES[name]['t_start'] for name in STAGE_ORDER[1:]]
print("Switch times (exact, hours):", [t / hour for t in SWITCH_TIMES])

dt = 20 * hour

t_f = CONCENTRATION_STAGES['LSWF']['t_end']


Switch times (exact, hours): [159.379368331, 381.360912196]


### Species Info

In [9]:
INIT_CALCITE = 10
CONST_H2O = 1.0
CONST_CO3 = 1.312e-4
CONST_CAOH = 5.248e-4
CONST_A = 1.014165e-3
CONST_N = 2.239614e-2
EQ_TEMPERATURE = 100.0 #C
EQ_PRESSURE = 1380 #psi
EQ_PH = 5
SSS = 350.0
SCS = 0.8
SURFACE_AREA = SSS * SCS

AQUEOUS_SPECIES = ['H+(aq)', 'OH-(aq)', 'Ca+2(aq)', 'HCO3-(aq)', 'CO3-2(aq)', 'Na+(aq)', 'Cl-(aq)']
SPECIES_KEYS = ['H+', 'OH-', 'Ca+2', 'HCO3-', 'CO3-2', 'Na+', 'Cl-']

SPECIES_FILE = {
    'H+(aq)': 'H_plus', 'OH-(aq)': 'OH_minus', 'Ca+2(aq)': 'Ca2_plus',
    'HCO3-(aq)': 'HCO3_minus', 'CO3-2(aq)': 'CO3_2minus', 'Na+(aq)': 'Na_plus', 'Cl-(aq)': 'Cl_minus',
}

LIQUID_SPECIES_NAMES = ['NH+', '-AH', '-ACa+', '-AMg+', 'A-', 'N']
LIQUID_SPECIES_FILE = {
    'NH+': 'NH_plus', '-AH': 'AH_minus', '-ACa+': 'ACa_plus',
    '-AMg+': 'AMg_plus', 'A-': 'A_minus', 'N': 'N',
}

AQUEOUS_PHASE_STR = "H+ OH- H2O(aq) CO3-2 SO4-2 Na+ Mg+2 Cl- Ca+2 HCO3- H2CO3 MgSO4(aq) NaSO4- CaSO4(aq) CaCl+ MgCl+"
MINERAL_PHASE_STR = "Calcite CaSO4- CaOH2+ CaCO3- CaOH CO3H CO3Ca+ CO3Mg+ CO3-"
LIQUID_PHASE_STR = "A- -AH -ACa+ -AMg+ N NH+"
DATABASE_FILE_PATH = os.path.join(Data_Path, 'Database.yaml') # The databse its required 

VALID_MODES = ('transport', 'equilibrium', 'transport+equilibrium')


### Plot Settings

In [10]:
AXIS_LABEL_FONTSIZE = 12
AXIS_LABEL_WEIGHT = 'bold'
TICK_LABEL_FONTSIZE = 12
TITLE_FONTSIZE = 12
LEGEND_FONTSIZE = 12

STAGE_COLOR_STOPS = {
    'HSWF': ('#595959', '#000000'),
    'MSWF': ('#7FA8FF', '#0033FF'),
    'LSWF': ('#FF8A8A', '#CC0000'),
}
STAGE_BASE_COLOR = {name: stops[1] for name, stops in STAGE_COLOR_STOPS.items()}


def _make_soft_to_solid_cmap(soft_hex, solid_hex):
    return mcolors.LinearSegmentedColormap.from_list(
        f"soft_to_solid_{solid_hex}", [soft_hex, solid_hex]
    )


STAGE_COLORMAPS = {
    name: _make_soft_to_solid_cmap(soft, solid)
    for name, (soft, solid) in STAGE_COLOR_STOPS.items()
}


In [11]:
# Functions
def _safe_fname(s):
    for ch in ['+', '-', ' ', '(', ')', '/', '\\', ':']:
        s = s.replace(ch, '_')
    while '__' in s:
        s = s.replace('__', '_')
    return s.strip('_')


def save_fig(fig, fname_stem, subfolder='Transport_Figures'):
    target_dir = Transport_Figures if subfolder == 'Transport_Figures' else Figure_Path
    path = os.path.join(target_dir, f"{_safe_fname(fname_stem)}.png")
    fig.savefig(path, dpi=200, bbox_inches='tight')
    print(f"  [figure saved] {path}")


def resolve_modes(run_mode, use_equilibrium):
    if run_mode not in VALID_MODES:
        raise ValueError(f"run_mode must be one of {VALID_MODES}, got '{run_mode}'")
    run_trans = run_mode in ('transport', 'transport+equilibrium')
    run_eq = use_equilibrium and run_mode in ('equilibrium', 'transport+equilibrium')
    return run_trans, run_eq

In [12]:

def bl_solve(theta, t_sec, u_total, phi, mu_w, mu_o, kr_theta0, kr_theta1, L_core, N=500, Sw_init=None):

    theta = float(np.clip(theta, 0.0, 1.0))

    krw0 = float((1 - theta) * kr_theta0['krw_max'] + theta * kr_theta1['krw_max'])
    kro0 = float((1 - theta) * kr_theta0['kro_max'] + theta * kr_theta1['kro_max'])
    Swr = float((1 - theta) * kr_theta0['swc'] + theta * kr_theta1['swc'])
    Sor = float((1 - theta) * kr_theta0['sor'] + theta * kr_theta1['sor'])
    nw = float((1 - theta) * kr_theta0['nw'] + theta * kr_theta1['nw'])
    no = float((1 - theta) * kr_theta0['no'] + theta * kr_theta1['no'])

    if Sw_init is None:
        Sw_init = Swr
    else:
        Sw_init = float(np.clip(Sw_init, Swr, 1.0 - Sor))

    S_max = 1.0 - Sor

    def Sw_norm(Sw):
        return (Sw - Swr) / (S_max - Swr)

    def krw(Sw):
        S_star = np.clip(Sw_norm(Sw), 0.0, 1.0)
        return krw0 * (S_star ** nw)

    def kro(Sw):
        S_star = np.clip(Sw_norm(Sw), 0.0, 1.0)
        return kro0 * ((1.0 - S_star) ** no)

    def fw(Sw):
        krw_val = krw(Sw)
        kro_val = kro(Sw)
        if krw_val == 0.0 and kro_val == 0.0:
            return 0.0
        return (krw_val / mu_w) / (krw_val / mu_w + kro_val / mu_o)

    Sw_vals = np.linspace(Swr, S_max, N, dtype=np.float64)
    fw_vals = np.array([fw(Sw) for Sw in Sw_vals], dtype=np.float64)
    dfw = np.gradient(fw_vals, Sw_vals)

    fw_init = fw(Sw_init)
    slopes = (fw_vals - fw_init) / (Sw_vals - Sw_init + 1e-12)
    mask_tangent = Sw_vals >= Sw_init
    if np.any(mask_tangent):
        slopes_masked = slopes.copy()
        slopes_masked[~mask_tangent] = -np.inf
        idx = np.argmax(slopes_masked)
    else:
        idx = np.argmax(slopes)
    Swf = float(Sw_vals[idx])
    slope = float(slopes[idx])
    v_shock = (u_total / phi) * slope

    mask = Sw_vals >= Swf
    Sw_cont = Sw_vals[mask]
    x_cont = (u_total / phi) * dfw[mask] * float(t_sec)
    order = np.argsort(x_cont)
    x_cont_sorted = np.asarray(x_cont[order], dtype=np.float64)
    Sw_cont_sorted = np.asarray(Sw_cont[order], dtype=np.float64)

    x_shock = v_shock * float(t_sec)
    x_grid = np.linspace(0.0, L_core, 500, dtype=np.float64)
    Sw_profile = Sw_init * np.ones_like(x_grid)

    if len(x_cont_sorted) > 1:
        Sw_interp = np.interp(x_grid, x_cont_sorted, Sw_cont_sorted,
                               left=S_max, right=Swf)
        mask = (x_grid <= x_shock) & (x_grid >= 0.0)
        Sw_profile[mask] = Sw_interp[mask]
        Sw_profile[x_grid > x_shock] = Sw_init
    else:
        Sw_profile[x_grid <= x_shock] = S_max
        Sw_profile[x_grid > x_shock] = Sw_init

    Sw_outlet = float(Sw_profile[-1])
    fw_outlet = fw(Sw_outlet)
    v_water_out = fw_outlet * u_total

    Sw_avg = np.trapz(Sw_profile, x_grid) / L_core
    recovery_frac = (Sw_avg - Swr) / (1.0 - Swr)
    recovery_percent = recovery_frac * 100.0

    return v_water_out, Sw_avg, recovery_percent

# Chemical Equilibrium

In [13]:
def _build_reaktoro_equilibrium_solver():
    db = Database.fromFile(DATABASE_FILE_PATH)
    solution = AqueousPhase(AQUEOUS_PHASE_STR)
    mineral = MineralPhase(MINERAL_PHASE_STR)
    liquid = LiquidPhase(LIQUID_PHASE_STR)
    system = ChemicalSystem(db, solution, mineral, liquid)

    specs = EquilibriumSpecs(system)
    specs.temperature()
    specs.pressure()
    specs.pH()
    solver = EquilibriumSolver(specs)

    conds = EquilibriumConditions(specs)
    conds.temperature(EQ_TEMPERATURE, "celsius")
    conds.pressure(EQ_PRESSURE, "psi")
    conds.pH(EQ_PH)

    return system, solver, conds

In [14]:
def compute_bps_for_brine(brine_comp, calcite_mass=INIT_CALCITE):
    system, solver, conds = _build_reaktoro_equilibrium_solver()

    state = ChemicalState(system)
    state.set("H2O(aq)", CONST_H2O, "kg")
    state.set("Calcite", calcite_mass, "kg")
    state.set("CO3-", CONST_CO3, "mol")
    state.set("CaOH", CONST_CAOH, "mol")
    state.set("A-", CONST_A, "mol")
    state.set("N", CONST_N, "mol")
    for sp, conc in brine_comp.items():
        if sp in ['Na', 'Ca', 'Mg', 'Cl', 'SO4', 'HCO3']:
            reaktoro_name = {'Na': 'Na+', 'Ca': 'Ca+2', 'Mg': 'Mg+2', 'Cl': 'Cl-', 'SO4': 'SO4-2', 'HCO3': 'HCO3-'}[sp]
            state.set(reaktoro_name, conc, "mol")
    solver.solve(state, conds)

    bps = (state.speciesAmount("NH+") / SURFACE_AREA) * (state.speciesAmount("CaSO4-") / SURFACE_AREA) + \
          (state.speciesAmount("NH+") / SURFACE_AREA) * (state.speciesAmount("CaCO3-") / SURFACE_AREA) + \
          (state.speciesAmount("A-") / SURFACE_AREA) * (state.speciesAmount("CO3Mg+") / SURFACE_AREA) + \
          (state.speciesAmount("A-") / SURFACE_AREA) * (state.speciesAmount("CO3Ca+") / SURFACE_AREA) + \
          (state.speciesAmount("-ACa+") / SURFACE_AREA) * (state.speciesAmount("CO3-") / SURFACE_AREA) + \
          (state.speciesAmount("-ACa+") / SURFACE_AREA) * (state.speciesAmount("CaSO4-") / SURFACE_AREA) + \
          (state.speciesAmount("-ACa+") / SURFACE_AREA) * (state.speciesAmount("CaCO3-") / SURFACE_AREA) + \
          (state.speciesAmount("-AMg+") / SURFACE_AREA) * (state.speciesAmount("CO3-") / SURFACE_AREA) + \
          (state.speciesAmount("-AMg+") / SURFACE_AREA) * (state.speciesAmount("CaSO4-") / SURFACE_AREA) + \
          (state.speciesAmount("-AMg+") / SURFACE_AREA) * (state.speciesAmount("CaCO3-") / SURFACE_AREA)
    return bps


def ionic_strength(brine):
    ions = {'Na': 1, 'Ca': 2, 'Mg': 2, 'Cl': -1, 'SO4': -2, 'HCO3': -1}
    is_val = 0.5 * sum(brine.get(ion, 0.0) * (ions[ion] ** 2) for ion in ions)
    return is_val


print("Computing BPS endpoints for HSWF, MSWF, LSWF ...")
BPS_HSWF = compute_bps_for_brine(HSWF)
BPS_MSWF = compute_bps_for_brine(MSWF)
BPS_LSWF = compute_bps_for_brine(LSWF)
print(f"BPS_HSWF = {BPS_HSWF:.6e}, BPS_MSWF = {BPS_MSWF:.6e}, BPS_LSWF = {BPS_LSWF:.6e}")

IS_HSWF = ionic_strength(HSWF)
IS_MSWF = ionic_strength(MSWF)
IS_LSWF = ionic_strength(LSWF)
print(f"IS_HSWF = {IS_HSWF:.6f}, IS_MSWF = {IS_MSWF:.6f}, IS_LSWF = {IS_LSWF:.6f}")



Computing BPS endpoints for HSWF, MSWF, LSWF ...
BPS_HSWF = 2.376062e-14, BPS_MSWF = 2.183709e-14, BPS_LSWF = 1.985448e-14
IS_HSWF = 1.510052, IS_MSWF = 1.258954, IS_LSWF = 0.011534


### Mesh function space and boundary steup

In [15]:
def create_mesh_and_domain():
    domain = Rectangle(Point(xmin, ymin), Point(xmax, ymax))
    mesh = RectangleMesh(Point(xmin, ymin), Point(xmax, ymax), 10, 16, "crossed")
    plot(mesh, title="Mesh")
    return {'mesh': mesh}


def create_function_spaces(mesh_dict, num_components):
    mesh = mesh_dict['mesh']
    V_list = [FunctionSpace(mesh, 'Lagrange', fs_order_transport) for _ in range(num_components)]
    return {'V_list': V_list, 'mesh': mesh, 'num_components': num_components}


def setup_boundary_parts(mesh_dict):
    mesh = mesh_dict['mesh']
    boundary_parts = MeshFunction("size_t", mesh, mesh.topology().dim() - 1)

    class BottomBoundary(SubDomain):
        def inside(self, x, on_boundary): return on_boundary and abs(x[1] - ymin) < 1E-14

    class RightBoundary(SubDomain):
        def inside(self, x, on_boundary): return on_boundary and abs(x[0] - xmax) < 1E-14

    class TopBoundary(SubDomain):
        def inside(self, x, on_boundary): return on_boundary and abs(x[1] - ymax) < 1E-14

    class LeftBoundary(SubDomain):
        def inside(self, x, on_boundary): return on_boundary and abs(x[0] - xmin) < 1E-14

    Gamma_1 = BottomBoundary(); Gamma_1.mark(boundary_parts, 1)
    Gamma_2 = RightBoundary(); Gamma_2.mark(boundary_parts, 2)
    Gamma_3 = TopBoundary(); Gamma_3.mark(boundary_parts, 3)
    Gamma_4 = LeftBoundary(); Gamma_4.mark(boundary_parts, 4)
    dx = Measure('dx')[boundary_parts]
    ds = Measure('ds')[boundary_parts]
    return {'boundary_parts': boundary_parts, 'dx': dx, 'ds': ds}


def setup_transport_boundary_conditions(space_dict, params_list, boundary_dict):
    V_list = space_dict['V_list']
    boundary_parts = boundary_dict['boundary_parts']
    bcs_list = []
    for V, params in zip(V_list, params_list):
        bc2 = DirichletBC(V, Constant(0.0), boundary_parts, 2)
        bc4 = DirichletBC(V, Constant(0.0), boundary_parts, 4)
        bcs_list.append([bc2, bc4])
    return {'bcs_list': bcs_list}


In [16]:
def setup_physical_parameters_multi(stage_name='HSWF'):
    stage_C0 = CONCENTRATION_STAGES[stage_name]['C0']
    component_order = ['Na', 'Ca', 'Mg', 'Cl', 'SO4', 'HCO3']
    params_list = []
    for name in component_order:
        c0 = stage_C0[name]
        alfa_L = ymax / PecletGlobal
        D_xx = alfa_L * u_total
        params_list.append({'c0': c0, 'alfa_L': alfa_L, 'D_xx': D_xx,
                             'c_in': c0, 'u_in': u_total, 'name': name})
    return params_list


def setup_initial_conditions_multi(space_dict, params_list):
    V_list = space_dict['V_list']
    c_0_list = []
    c_1_list = []
    for V, params in zip(V_list, params_list):
        c0 = params['c0']
        c_0 = Expression('c0', c0=c0, degree=1)
        c_1 = project(c_0, V)
        c_0_list.append(c_0)
        c_1_list.append(c_1)
    return {'c_0_list': c_0_list, 'c_1_list': c_1_list}


def reset_concentrations_for_stage(space_dict, params_list, init_dict, stage_name):
    V_list = space_dict['V_list']
    c_1_list = init_dict['c_1_list']
    stage_C0 = CONCENTRATION_STAGES[stage_name]['C0']
    for V, params, c_1 in zip(V_list, params_list, c_1_list):
        name = params['name']
        new_c0 = stage_C0[name]
        params['c0'] = new_c0
        params['c_in'] = new_c0
        c_0_new = Expression('c0', c0=new_c0, degree=1)
        c_1.assign(project(c_0_new, V))
    print(f"  >> Switched inlet/initial concentrations to stage '{stage_name}': "
          f"{ {n: stage_C0[n] for n in stage_C0} }")



# Equilibrium Function


In [17]:

def equilibrium_update_dof(na_func, ca_func, mg_func, cl_func, so4_func, hco3_func, calcite_func, V):
    system, solver, conds = _build_reaktoro_equilibrium_solver()

    na_dof = na_func.vector().get_local()
    ca_dof = ca_func.vector().get_local()
    mg_dof = mg_func.vector().get_local()
    cl_dof = cl_func.vector().get_local()
    so4_dof = so4_func.vector().get_local()
    hco3_dof = hco3_func.vector().get_local()
    calcite_dof = calcite_func.vector().get_local()

    na_updated_vals = np.zeros_like(na_dof)
    ca_updated_vals = np.zeros_like(na_dof)
    mg_updated_vals = np.zeros_like(na_dof)
    cl_updated_vals = np.zeros_like(na_dof)
    so4_updated_vals = np.zeros_like(na_dof)
    hco3_updated_vals = np.zeros_like(na_dof)
    calcite_updated_vals = np.zeros_like(na_dof)
    a_minus_vals = np.zeros_like(na_dof)
    n_vals = np.zeros_like(na_dof)
    nh_plus_vals = np.zeros_like(na_dof)
    ah_minus_vals = np.zeros_like(na_dof)
    aca_plus_vals = np.zeros_like(na_dof)
    amg_plus_vals = np.zeros_like(na_dof)
    is_values = np.zeros_like(na_dof)
    bps_values = np.zeros_like(na_dof)
    pH_values = np.zeros_like(na_dof)
    species_arrays = {key: np.zeros_like(na_dof) for key in SPECIES_KEYS}

    for i in range(len(na_dof)):
        state = ChemicalState(system)
        state.set("H2O(aq)", CONST_H2O, "kg")
        state.set("Calcite", max(float(calcite_dof[i]), 0.0), "kg")
        state.set("CO3-", CONST_CO3, "mol")
        state.set("CaOH", CONST_CAOH, "mol")
        state.set("A-", CONST_A, "mol")
        state.set("N", CONST_N, "mol")
        state.set("Na+", max(float(na_dof[i]), 0.0), "mol")
        state.set("Ca+2", max(float(ca_dof[i]), 0.0), "mol")
        state.set("Mg+2", max(float(mg_dof[i]), 0.0), "mol")
        state.set("Cl-", max(float(cl_dof[i]), 0.0), "mol")
        state.set("SO4-2", max(float(so4_dof[i]), 0.0), "mol")
        state.set("HCO3-", max(float(hco3_dof[i]), 0.0), "mol")
        solver.solve(state, conds)
        props = AqueousProps(state)

        na_updated_vals[i] = float(state.speciesAmount("Na+"))
        ca_updated_vals[i] = float(state.speciesAmount("Ca+2"))
        mg_updated_vals[i] = float(state.speciesAmount("Mg+2"))
        cl_updated_vals[i] = float(state.speciesAmount("Cl-"))
        so4_updated_vals[i] = float(state.speciesAmount("SO4-2"))
        hco3_updated_vals[i] = float(state.speciesAmount("HCO3-"))
        calcite_updated_vals[i] = float(state.speciesMass("Calcite"))
        a_minus_vals[i] = float(state.speciesAmount("A-"))
        n_vals[i] = float(state.speciesAmount("N"))
        nh_plus_vals[i] = float(state.speciesAmount("NH+"))
        ah_minus_vals[i] = float(state.speciesAmount("-AH"))
        aca_plus_vals[i] = float(state.speciesAmount("-ACa+"))
        amg_plus_vals[i] = float(state.speciesAmount("-AMg+"))
        is_values[i] = float(props.ionicStrength())
        pH_values[i] = float(props.pH())
        for key in SPECIES_KEYS:
            species_arrays[key][i] = float(props.speciesMolality(key))

        Bond_Product_Sum = sum([
            (state.speciesAmount("NH+") / SURFACE_AREA) * (state.speciesAmount("CaSO4-") / SURFACE_AREA),
            (state.speciesAmount("NH+") / SURFACE_AREA) * (state.speciesAmount("CaCO3-") / SURFACE_AREA),
            (state.speciesAmount("A-") / SURFACE_AREA) * (state.speciesAmount("CO3Mg+") / SURFACE_AREA),
            (state.speciesAmount("A-") / SURFACE_AREA) * (state.speciesAmount("CO3Ca+") / SURFACE_AREA),
            (state.speciesAmount("-ACa+") / SURFACE_AREA) * (state.speciesAmount("CO3-") / SURFACE_AREA),
            (state.speciesAmount("-ACa+") / SURFACE_AREA) * (state.speciesAmount("CaSO4-") / SURFACE_AREA),
            (state.speciesAmount("-ACa+") / SURFACE_AREA) * (state.speciesAmount("CaCO3-") / SURFACE_AREA),
            (state.speciesAmount("-AMg+") / SURFACE_AREA) * (state.speciesAmount("CO3-") / SURFACE_AREA),
            (state.speciesAmount("-AMg+") / SURFACE_AREA) * (state.speciesAmount("CaSO4-") / SURFACE_AREA),
            (state.speciesAmount("-AMg+") / SURFACE_AREA) * (state.speciesAmount("CaCO3-") / SURFACE_AREA),
        ])
        bps_values[i] = Bond_Product_Sum

    def _make_func(vals):
        f = Function(V)
        f.vector().set_local(vals)
        f.vector().apply("insert")
        return f

    na_out = _make_func(na_updated_vals)
    ca_out = _make_func(ca_updated_vals)
    mg_out = _make_func(mg_updated_vals)
    cl_out = _make_func(cl_updated_vals)
    so4_out = _make_func(so4_updated_vals)
    hco3_out = _make_func(hco3_updated_vals)
    calcite_out = _make_func(calcite_updated_vals)
    a_minus_out = _make_func(a_minus_vals)
    n_out = _make_func(n_vals)
    nh_plus_out = _make_func(nh_plus_vals)
    ah_minus_out = _make_func(ah_minus_vals)
    aca_plus_out = _make_func(aca_plus_vals)
    amg_plus_out = _make_func(amg_plus_vals)
    is_func = _make_func(is_values)
    bps_func = _make_func(bps_values)
    ph_func = _make_func(pH_values)

    species_funcs = {}
    for sp_label, key in zip(AQUEOUS_SPECIES, SPECIES_KEYS):
        species_funcs[sp_label] = _make_func(species_arrays[key])

    liquid_funcs = {
        'NH+': nh_plus_out, '-AH': ah_minus_out,
        '-ACa+': aca_plus_out, '-AMg+': amg_plus_out,
        'A-': a_minus_out, 'N': n_out,
    }

    return (na_out, ca_out, mg_out, cl_out, so4_out, hco3_out, calcite_out,
            liquid_funcs, species_funcs, ph_func, is_func, bps_func)


### Wettability/ Relative permeability 

In [18]:
def _kr_corey_curve(sw, params):
    swd = (sw - params['swc']) / (1 - params['sor'] - params['swc'])
    swd = np.clip(swd, 0.0, 1.0)
    kro = params['kro_max'] * ((1 - swd) ** params['no'])
    krw = params['krw_max'] * (swd ** params['nw'])
    return kro, krw


def compute_per_stage_theta(time_array, data_array):
    theta_array = np.zeros_like(data_array, dtype=float)
    for stage_idx, stage_name in enumerate(STAGE_ORDER):
        if stage_idx == 0:
            si = 0
        else:
            si = int(np.argmin(np.abs(time_array - SWITCH_TIMES[stage_idx - 1])))
        if stage_idx == len(STAGE_ORDER) - 1:
            ei = len(time_array) - 1
        else:
            ei = int(np.argmin(np.abs(time_array - SWITCH_TIMES[stage_idx])))

        window = data_array[si:ei + 1]
        data_min = float(np.min(window))
        data_max = float(np.max(window))
        denom = data_max - data_min

        for i in range(si, ei + 1):
            if abs(denom) < 1e-30:
                theta_array[i] = 0.5
            else:
                theta_array[i] = (data_array[i] - data_min) / denom
            theta_array[i] = np.clip(theta_array[i], 0.0, 1.0)
    return theta_array


def interpolate_kr(sw, theta, kr0, kr1):
    theta = float(np.clip(theta, 0.0, 1.0))
    krw0 = (1 - theta) * kr0['krw_max'] + theta * kr1['krw_max']
    kro0 = (1 - theta) * kr0['kro_max'] + theta * kr1['kro_max']
    swc = (1 - theta) * kr0['swc'] + theta * kr1['swc']
    sor = (1 - theta) * kr0['sor'] + theta * kr1['sor']
    nw = (1 - theta) * kr0['nw'] + theta * kr1['nw']
    no = (1 - theta) * kr0['no'] + theta * kr1['no']

    swd = (sw - swc) / (1 - sor - swc)
    swd = np.clip(swd, 0.0, 1.0)
    kro = kro0 * ((1 - swd) ** no)
    krw = krw0 * (swd ** nw)
    return kro, krw


def get_stage_boundaries(time_array):
    bounds = []
    for stage_idx, stage_name in enumerate(STAGE_ORDER):
        si = 0 if stage_idx == 0 else int(np.argmin(np.abs(time_array - SWITCH_TIMES[stage_idx - 1])))
        ei = (len(time_array) - 1 if stage_idx == len(STAGE_ORDER) - 1
              else int(np.argmin(np.abs(time_array - SWITCH_TIMES[stage_idx]))))
        bounds.append((stage_name, si, ei))
    return bounds


In [19]:

def normalize_0_to_1(data, data_min, data_max):
    if data_max == data_min:
        return np.full_like(data, 0.5)
    else:
        return (data - data_min) / (data_max - data_min)


def XYplot(x, y, x_min, x_max, x_label, y_label, color, fname_stem=None, subfolder='Transport_Figures'):
    fig = plt.figure(figsize=(8, 6))
    plt.plot(x, y, color, linewidth=2)
    plt.xlim(x_min, x_max)
    plt.xlabel(x_label, fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    plt.ylabel(y_label, fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    plt.xticks(fontsize=TICK_LABEL_FONTSIZE)
    plt.yticks(fontsize=TICK_LABEL_FONTSIZE)
    plt.grid(False)
    plt.tight_layout()
    stem = fname_stem if fname_stem else y_label
    save_fig(fig, stem, subfolder=subfolder)
    plt.show(block=False)
    plt.pause(0.1)


# BPS vs TIME PLOTTING 

In [20]:

def plot_bps_time(time_h, bps_data, t_start_h, t_end_h,
                  y_top_max=None, y_bot_max=None,
                  fname_stem='BPS_vs_Time_broken'):

    time_sliced = time_h[1:]
    bps_sliced  = bps_data[1:]

    if y_top_max is None:
        y_top_max = 1.15 * float(np.max(bps_sliced)) if np.max(bps_sliced) > 0 else 1.0
    if y_bot_max is None:
        y_bot_max = 5.0 * max(BPS_HSWF, BPS_MSWF, BPS_LSWF)

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, sharex=True,
        figsize=(10, 8),
        gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.08}
    )

    ax_top.plot(time_sliced, bps_sliced, 'm-', linewidth=2.0)
    ax_top.set_ylim(0, y_top_max)
    ax_top.set_ylabel("Bond Product Sum\n(mol/m²)²",
                      fontsize=AXIS_LABEL_FONTSIZE,
                      fontweight=AXIS_LABEL_WEIGHT)
    ax_top.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    ax_top.grid(False)
    ax_top.spines['bottom'].set_visible(False)
    ax_top.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    for t_sw in SWITCH_TIMES:
        ax_top.axvline(t_sw / hour, color='gray', linestyle=':', alpha=0.6, linewidth=1.0)
        ax_bot.axvline(t_sw / hour, color='gray', linestyle=':', alpha=0.6, linewidth=1.0)

    ax_bot.plot(time_sliced, bps_sliced, 'm-', linewidth=2.0)
    ax_bot.set_ylim(0, y_bot_max)
    ax_bot.set_xlim(t_start_h, t_end_h)
    ax_bot.set_xlabel("Time [hours]",
                      fontsize=AXIS_LABEL_FONTSIZE,
                      fontweight=AXIS_LABEL_WEIGHT)
    ax_bot.set_ylabel("BPS (zoom)\n(mol/m²)²",
                      fontsize=AXIS_LABEL_FONTSIZE - 4,
                      fontweight=AXIS_LABEL_WEIGHT)
    ax_bot.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    ax_bot.grid(False)
    ax_bot.spines['top'].set_visible(False)

    ax_bot.axhline(BPS_HSWF, color='black',  linestyle='--', linewidth=1.2,
                   label=f'BPS$_{{HSWF}}$ = {BPS_HSWF:.2e}')
    ax_bot.axhline(BPS_MSWF, color='#0033FF', linestyle='--', linewidth=1.2,
                   label=f'BPS$_{{MSWF}}$ = {BPS_MSWF:.2e}')
    ax_bot.axhline(BPS_LSWF, color='#CC0000', linestyle='--', linewidth=1.2,
                   label=f'BPS$_{{LSWF}}$ = {BPS_LSWF:.2e}')
    ax_bot.legend(loc='upper right', fontsize=LEGEND_FONTSIZE - 8, frameon=True)

    d_break = 0.015
    kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False, linewidth=1.2)
    ax_top.plot((-d_break, +d_break), (-d_break, +d_break), **kwargs)
    ax_top.plot((1 - d_break, 1 + d_break), (-d_break, +d_break), **kwargs)
    kwargs.update(transform=ax_bot.transAxes)
    ax_bot.plot((-d_break, +d_break), (1 - d_break, 1 + d_break), **kwargs)
    ax_bot.plot((1 - d_break, 1 + d_break), (1 - d_break, 1 + d_break), **kwargs)

    plt.tight_layout()
    save_fig(fig, fname_stem, subfolder='Transport_Figures')
    plt.show(block=False)
    plt.pause(0.1)


In [21]:

def plot_bps_time_symlog(time_h, bps_data, t_start_h, t_end_h,
                          floor=1e-16,
                          ylabel_fontsize=15,
                          fname_stem='BPS_vs_Time_symlog'):

    time_sliced = time_h[1:]
    bps_sliced  = np.asarray(bps_data[1:], dtype=float)

    # Plot-only copy: no zeros / negatives allowed on a log axis
    bps_plot = np.maximum(bps_sliced, floor)

    fig, ax = plt.subplots(figsize=(11, 7), facecolor='white')
    ax.set_facecolor('white')
    ax.plot(time_sliced, bps_plot, 'm-', linewidth=2.2, zorder=3)

    # ---- log y-axis ----
    ax.set_yscale('log')
    ax.set_ylim(floor / 10.0, 3.0 * float(np.max(bps_plot)))
    ax.set_xlim(t_start_h, t_end_h)

    # ---- axis labels ----
    ax.set_xlabel("Time [hours]",
                  fontsize=AXIS_LABEL_FONTSIZE,
                  fontweight=AXIS_LABEL_WEIGHT)
    ax.set_ylabel("Bond Product Sum (mol/m²)²",
                  fontsize=ylabel_fontsize,
                  fontweight=AXIS_LABEL_WEIGHT)
    ax.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    ax.tick_params(axis='y', which='minor', labelleft=False)
    ax.grid(False)

    plt.tight_layout()
    fig.savefig(os.path.join(Transport_Figures, f"{_safe_fname(fname_stem)}.png"),
                dpi=200, bbox_inches='tight', facecolor='white')
    print(f"  [figure saved] {fname_stem}")
    plt.show(block=False)
    plt.pause(0.1)

    # ---- CSV dump (raw data, not clipped) ----
    bps_vs_time_df = pd.DataFrame({'Time [h]': time_sliced, 'BPS [-]': bps_sliced})
    bps_vs_time_csv_path = os.path.join(Data_Path, 'BPS_vs_Time.csv')
    bps_vs_time_df.to_csv(bps_vs_time_csv_path, index=False)
    print(f"  [BPS vs Time CSV saved] {bps_vs_time_csv_path}")

def plot_normalized_bps(time, time_h, bps_data, t_start_h, t_end_h):
    bps_norm = compute_per_stage_theta(time, bps_data)
    time_sliced = time_h[1:]
    bps_norm_sliced = bps_norm[1:]
    fig = plt.figure(figsize=(8, 6))
    plt.plot(time_sliced, bps_norm_sliced, 'c-', linewidth=2)
    plt.xlim(t_start_h, t_end_h)
    plt.ylim(-0.05, 1.05)
    plt.xlabel("Time [hours]", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    plt.ylabel("Normalized Bond Product Sum (\u03b8)", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    plt.xticks(fontsize=TICK_LABEL_FONTSIZE)
    plt.yticks(fontsize=TICK_LABEL_FONTSIZE)
    plt.grid(False)
    plt.tight_layout()
    save_fig(fig, 'Normalized_BPS_vs_Time', subfolder='Transport_Figures')
    plt.show(block=False)
    plt.pause(0.1)


def plot_normalized_is(time, time_h, is_data, t_start_h, t_end_h):
    is_max = np.max(is_data)
    is_min = np.min(is_data)
    is_norm = (is_max - is_data) / (is_max - is_min) if is_max != is_min else np.ones_like(is_data)
    time_sliced = time_h[1:]
    is_norm_sliced = is_norm[1:]
    fig = plt.figure(figsize=(8, 6))
    plt.plot(time_sliced, is_norm_sliced, 'b-', linewidth=2)
    plt.xlim(t_start_h, t_end_h)
    plt.ylim(-0.05, 1.05)
    plt.xlabel("Time [hours]", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    plt.ylabel("Normalized Ionic Strength", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    plt.xticks(fontsize=TICK_LABEL_FONTSIZE)
    plt.yticks(fontsize=TICK_LABEL_FONTSIZE)
    plt.grid(False)
    plt.tight_layout()
    save_fig(fig, 'Normalized_IonicStrength_vs_Time', subfolder='Transport_Figures')
    plt.show(block=False)
    plt.pause(0.1)


def print_per_stage_normalization_info(time, bps_data, is_data):
    theta0 = "\u03b8=0"
    theta1 = "\u03b8=1"
    hdr = (f"  {'Stage':<8} {'t_start[h]':>11} {'t_end[h]':>11} "
           f"{'BPS^LS(' + theta0 + ')':>15} {'BPS^HS(' + theta1 + ')':>15} "
           f"{'IS^LS(' + theta0 + ')':>13} {'IS^HS(' + theta1 + ')':>13}")
    sep = "  " + "-" * (len(hdr) - 2)
    print(f"\n{'=' * 90}")
    print(f"{'=' * 90}")
    print(hdr)
    print(sep)
    for stage_idx, stage_name in enumerate(STAGE_ORDER):
        si = 0 if stage_idx == 0 else int(np.argmin(np.abs(time - SWITCH_TIMES[stage_idx - 1])))
        ei = (len(time) - 1 if stage_idx == len(STAGE_ORDER) - 1
              else int(np.argmin(np.abs(time - SWITCH_TIMES[stage_idx]))))
        print(f"  {stage_name:<8} {time[si] / hour:>11.2f} {time[ei] / hour:>11.2f} "
              f"{bps_data[si]:>15.6e} {bps_data[ei]:>15.6e} "
              f"{is_data[si]:>13.6e} {is_data[ei]:>13.6e}")
    print(f"{'=' * 90}\n")


def plot_wettability_altered_kr(global_eq):
    if not global_eq:
        print("No equilibrium results available; skipping wettability-altered kr plots.")
        return

    print(f"\n{'=' * 60}")
    print("  Wettability-Altered Relative Permeability")
    print("  HSWF: HSWF<->MSWF   |   MSWF: MSWF<->LSWF   |   LSWF: LSWF constant")
    print(f"{'=' * 60}")

    time = global_eq['time']
    sw = np.linspace(0.0, 1.0, 500)
    bps_data = global_eq['bps_mid_time']
    is_data = global_eq['is_mid_time']
    time_h = time / hour
    t_start_h = time_h[1]
    t_end_h = time_h[-1]

    # --- MAIN: single-panel symlog BPS plot (shows baseline AND peaks) ---
    plot_bps_time_symlog(time_h, bps_data, t_start_h, t_end_h)

    # --- Optional alternatives (uncomment if you prefer them) ---
    # plot_bps_time(time_h, bps_data, t_start_h, t_end_h)   # broken-axis version

    plot_normalized_bps(time, time_h, bps_data, t_start_h, t_end_h)

    fig_is = plt.figure(figsize=(8, 6))
    plt.plot(time_h[1:], is_data[1:], 'g-', linewidth=2)
    plt.xlim(t_start_h, t_end_h)
    plt.xlabel("Time [hours]", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    plt.ylabel("Ionic Strength (molal)", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    plt.xticks(fontsize=TICK_LABEL_FONTSIZE)
    plt.yticks(fontsize=TICK_LABEL_FONTSIZE)
    plt.grid(False)
    plt.tight_layout()
    save_fig(fig_is, 'IonicStrength_vs_Time', subfolder='Transport_Figures')
    plt.show(block=False)
    plt.pause(0.1)

    plot_normalized_is(time, time_h, is_data, t_start_h, t_end_h)

    bps_theta = compute_per_stage_theta(time, bps_data)
    is_theta = compute_per_stage_theta(time, is_data)

    nt = len(time) - 1
    stage_bounds = get_stage_boundaries(time)
    idx_to_stage = {}
    for name, si, ei in stage_bounds:
        for i in range(si, ei + 1):
            idx_to_stage[i] = name

    for theta_arr, var_label, fname_tag in [
        (bps_theta, 'BPS', 'kr_wettability_altered_BPS_stagewise'),
        (is_theta, 'IS', 'kr_wettability_altered_IS_stagewise'),
    ]:
        fig = plt.figure(figsize=(18, 14))
        ax = fig.add_axes([0.08, 0.05, 0.88, 0.88])
        ax.set_xlim(0, 0.5)
        ax.set_ylim(0, 0.08)
        ax.set_xlabel('Water Saturation $S_w$',
                       fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
        ax.set_ylabel('Relative Permeability $k_r$',
                       fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
        ax.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
        ax.grid(False)

        for i in range(1, nt + 1):
            theta_i = float(np.clip(theta_arr[i], 0.0, 1.0))
            stage_name = idx_to_stage[i]
            if stage_name == 'HSWF':
                kr0, kr1 = KR_HSWF, KR_MSWF
            elif stage_name == 'MSWF':
                kr0, kr1 = KR_MSWF, KR_LSWF
            else:
                kr0, kr1 = KR_LSWF, KR_LSWF

            kro_i, krw_i = interpolate_kr(sw, theta_i, kr0, kr1)

            cmap = STAGE_COLORMAPS[stage_name]
            si, ei = None, None
            for name, s, e in stage_bounds:
                if name == stage_name:
                    si, ei = s, e
                    break
            frac = (i - si) / max(ei - si, 1) if ei > si else 0.5
            c = cmap(frac)

            ax.plot(sw, krw_i, color=c, lw=2, ls='-')
            ax.plot(sw, kro_i, color=c, lw=2, ls=':')

        from matplotlib.lines import Line2D
        legend_handles = []
        for name in STAGE_ORDER:
            base_color = STAGE_BASE_COLOR[name]
            legend_handles.append(
                Line2D([0], [0], color=base_color, lw=4, label=name))
        ax.legend(handles=legend_handles, loc='upper right',
                  fontsize=LEGEND_FONTSIZE, frameon=True)

        save_fig(fig, fname_tag, subfolder='Transport_Figures')
        plt.show(block=False)
        plt.pause(0.1)

    pd.DataFrame({'time_s': time, 'theta_bps_perstage': bps_theta}).to_csv(
        os.path.join(Transport_Path, 'theta_from_bps_perstage.csv'), index=False)
    pd.DataFrame({'time_s': time, 'theta_is_perstage': is_theta}).to_csv(
        os.path.join(Transport_Path, 'theta_from_is_perstage.csv'), index=False)


# TRANSPORT SOLVER (MAIN LOOP)


In [22]:

def solve_transport_problem(space_dict, boundary_dict, transport_bc_dict,
                             init_dict, params_list, run_trans, run_eq):
    c_1_list = init_dict['c_1_list']
    V_list = space_dict['V_list']
    bcs_list = transport_bc_dict['bcs_list']
    num_components = space_dict['num_components']
    dx = boundary_dict['dx']
    ds = boundary_dict['ds']
    name_to_idx = {p['name']: idx for idx, p in enumerate(params_list)}

    if run_trans:
        c_files = [File(os.path.join(Transport_Path, f"concentration_{p['name']}.pvd")) for p in params_list]
    if run_eq:
        species_files = {sp: File(os.path.join(Transport_Path, f"{SPECIES_FILE[sp]}.pvd")) for sp in AQUEOUS_SPECIES}
        pH_file = File(os.path.join(Transport_Path, 'pH.pvd'))
        is_file = File(os.path.join(Transport_Path, 'ionic_strength.pvd'))
        bps_file = File(os.path.join(Transport_Path, 'bond_product_sum.pvd'))
        calcite_file = File(os.path.join(Transport_Path, 'Calcite.pvd'))
        liquid_files = {sp: File(os.path.join(Transport_Path, f"{LIQUID_SPECIES_FILE[sp]}.pvd"))
                         for sp in LIQUID_SPECIES_NAMES}

    c_list = [Function(V) for V in V_list]

    time_list = []
    vel_water_list = []
    recovery_bps_list = []
    recovery_is_list = []
    species_results = {params['name']: {'c_t_in': [], 'c_t_out': []} for params in params_list}
    bps_mid_list = []
    is_mid_list = []

    current_stage = STAGE_ORDER[0]
    pending_switches = list(zip(SWITCH_TIMES, STAGE_ORDER[1:]))

    if run_eq:
        V_eq = V_list[0]
        calcite_func = project(Expression('c0', c0=INIT_CALCITE, degree=1), V_eq)
    else:
        calcite_func = None

    eq_result_shared = None
    t = 0.0
    step_counter = 0

    print("=== Starting time stepping (three-stage injection) ===")
    print(f"Stage '{current_stage}' active from t={CONCENTRATION_STAGES[current_stage]['t_start'] / hour:.2f}h")

    # --- Initial BL solve at t=0 (Sw_init = Swr) ---
    v_water_0, Sw_avg_prev, rec0 = bl_solve(0.0, 0.0, u_total, phi, mu_w, mu_o,
                                             KR_HSWF, KR_HSWF, L, Sw_init=None)
    Swr0 = KR_HSWF['swc']

    time_list.append(0.0)
    vel_water_list.append(v_water_0)
    recovery_bps_list.append(0.0)
    recovery_is_list.append(0.0)
    cumulative_recovery_bps = 0.0
    cumulative_recovery_is = 0.0
    for name in species_results:
        species_results[name]['c_t_in'].append(0.0)
        species_results[name]['c_t_out'].append(0.0)
    if run_eq:
        bps_mid_list.append(0.0)
        is_mid_list.append(0.0)

    v_water_current = v_water_0

    while t < CONCENTRATION_STAGES['LSWF']['t_end'] - 1e-9:
        next_switch = pending_switches[0][0] if pending_switches else CONCENTRATION_STAGES['LSWF']['t_end']
        dt_step = min(dt, next_switch - t)
        if dt_step < 1e-6:
            if pending_switches and abs(t - pending_switches[0][0]) < 1e-6:
                _, new_stage = pending_switches.pop(0)
                reset_concentrations_for_stage(space_dict, params_list, init_dict, new_stage)
                current_stage = new_stage
                print(f"  >> t = {t / hour:.2f}h: entering stage '{current_stage}' "
                      f"(active until t={CONCENTRATION_STAGES[current_stage]['t_end'] / hour:.2f}h)")
            continue

        if run_trans:
            for idx, params in enumerate(params_list):
                alfa_L = params['alfa_L']
                V = V_list[idx]
                c_1 = c_1_list[idx]
                vel_y = Constant(v_water_current)
                D_xx = alfa_L * v_water_current
                D_yy = alfa_L * v_water_current
                D = as_matrix([[Constant(D_xx), Constant(D_xy)],
                                [Constant(D_xy), Constant(D_yy)]])
                u_vec = as_vector([Constant(0.0), vel_y])
                c = TrialFunction(V)
                w = TestFunction(V)
                a_t = (R * c * w * dx
                       + dt_step * inner(nabla_grad(w), dot(D, nabla_grad(c)) - u_vec * c) * dx
                       + dt_step * vel_y * c * w * ds(3))
                L_t = (R * c_1 + dt_step * q_c) * w * dx
                A_t = assemble(a_t)
                b_t = assemble(L_t)
                for bc in bcs_list[idx]:
                    bc.apply(A_t, b_t)
                solve(A_t, c_list[idx].vector(), b_t)

            if run_eq:
                na_idx = name_to_idx['Na']
                ca_idx = name_to_idx['Ca']
                mg_idx = name_to_idx['Mg']
                cl_idx = name_to_idx['Cl']
                so4_idx = name_to_idx['SO4']
                hco3_idx = name_to_idx['HCO3']
                (na_eq, ca_eq, mg_eq, cl_eq, so4_eq, hco3_eq, calcite_eq,
                 liquid_funcs, species_funcs, ph_func, is_func, bps_func) = \
                    equilibrium_update_dof(
                        c_list[na_idx], c_list[ca_idx], c_list[mg_idx],
                        c_list[cl_idx], c_list[so4_idx], c_list[hco3_idx],
                        calcite_func, V_eq)

                # ==========================================================
                # [DIAGNOSTIC]  Compare initial in-situ BPS with static
                # BPS_HSWF. Runs only on the very first equilibrium update.
                # ==========================================================
                if step_counter == 0:
                    bps_inlet  = bps_func(radius, 0.0)
                    bps_mid_v  = bps_func(radius, ymax / 2)
                    bps_outlet = bps_func(radius, ymax)
                    
                c_list[na_idx].assign(na_eq); c_1_list[na_idx].assign(na_eq)
                c_list[ca_idx].assign(ca_eq); c_1_list[ca_idx].assign(ca_eq)
                c_list[mg_idx].assign(mg_eq); c_1_list[mg_idx].assign(mg_eq)
                c_list[cl_idx].assign(cl_eq); c_1_list[cl_idx].assign(cl_eq)
                c_list[so4_idx].assign(so4_eq); c_1_list[so4_idx].assign(so4_eq)
                c_list[hco3_idx].assign(hco3_eq); c_1_list[hco3_idx].assign(hco3_eq)
                calcite_func.assign(calcite_eq)
                eq_result_shared = {
                    'species_funcs': species_funcs,
                    'liquid_funcs': liquid_funcs,
                    'ph_func': ph_func, 'is_func': is_func,
                    'bps_func': bps_func, 'a_minus_func': liquid_funcs['A-'],
                }
                for sp in AQUEOUS_SPECIES:
                    sp_h = project(species_funcs[sp], V_eq)
                    sp_h.rename(f"{sp}", f"{sp}")
                    species_files[sp] << (sp_h, t + dt_step)
                for sp_name in LIQUID_SPECIES_NAMES:
                    sp_h = project(liquid_funcs[sp_name], V_eq)
                    sp_h.rename(sp_name, sp_name)
                    liquid_files[sp_name] << (sp_h, t + dt_step)
                ph_h = project(ph_func, V_eq); ph_h.rename("pH", "pH"); pH_file << (ph_h, t + dt_step)
                is_h = project(is_func, V_eq); is_h.rename("Ionic_Strength", "Ionic_Strength"); is_file << (is_h, t + dt_step)
                bps_h = project(bps_func, V_eq); bps_h.rename("Bond_Product_Sum", "Bond_Product_Sum"); bps_file << (bps_h, t + dt_step)
                calc_h = project(calcite_func, V_eq); calc_h.rename("Calcite", "Calcite"); calcite_file << (calc_h, t + dt_step)
            else:
                for idx in range(num_components):
                    c_1_list[idx].assign(c_list[idx])

            for idx, params in enumerate(params_list):
                name = params['name']
                ch = project(c_list[idx], V_list[idx])
                ch.rename(f"conc_{name}", f"conc_{name}")
                c_files[idx] << (ch, t + dt_step)

        t += dt_step
        step_counter += 1
        time_list.append(t)

        if run_trans:
            for idx, params in enumerate(params_list):
                name = params['name']
                c = c_list[idx]
                species_results[name]['c_t_in'].append(c(radius, 0))
                species_results[name]['c_t_out'].append(c(radius, ymax))

        if run_eq and eq_result_shared is not None:
            bps_mid = eq_result_shared['bps_func'](radius, ymax / 2)
            is_mid = eq_result_shared['is_func'](radius, ymax / 2)
            bps_mid_list.append(bps_mid)
            is_mid_list.append(is_mid)

            if current_stage == 'HSWF':
                bps_start = BPS_HSWF; bps_end = BPS_MSWF
                is_start = IS_HSWF; is_end = IS_MSWF
                kr_theta0 = KR_HSWF; kr_theta1 = KR_MSWF
            elif current_stage == 'MSWF':
                bps_start = BPS_MSWF; bps_end = BPS_LSWF
                is_start = IS_MSWF; is_end = IS_LSWF
                kr_theta0 = KR_MSWF; kr_theta1 = KR_LSWF
            else:
                bps_start = BPS_LSWF; bps_end = BPS_LSWF
                is_start = IS_LSWF; is_end = IS_LSWF
                kr_theta0 = KR_LSWF; kr_theta1 = KR_LSWF

            denom_bps = bps_end - bps_start
            theta_bps = (bps_mid - bps_start) / denom_bps if abs(denom_bps) > 1e-30 else 0.5
            theta_bps = np.clip(theta_bps, 0.0, 1.0)

            denom_is = is_end - is_start
            theta_is = (is_mid - is_start) / denom_is if abs(denom_is) > 1e-30 else 0.5
            theta_is = np.clip(theta_is, 0.0, 1.0)
        else:
            theta_bps = 0.5
            theta_is = 0.5
            kr_theta0 = KR_LSWF
            kr_theta1 = KR_LSWF

        v_water_bps, Sw_avg_new_bps, _ = bl_solve(theta_bps, t, u_total, phi, mu_w, mu_o,
                                                   kr_theta0, kr_theta1, L, Sw_init=Sw_avg_prev)
        _, Sw_avg_new_is, _ = bl_solve(theta_is, t, u_total, phi, mu_w, mu_o,
                                       kr_theta0, kr_theta1, L, Sw_init=Sw_avg_prev)

        delta_rec_bps = (Sw_avg_new_bps - Sw_avg_prev) / (1.0 - Swr0) * 100.0
        delta_rec_is  = (Sw_avg_new_is - Sw_avg_prev) / (1.0 - Swr0) * 100.0

        cumulative_recovery_bps += delta_rec_bps
        cumulative_recovery_is  += delta_rec_is

        recovery_bps_list.append(cumulative_recovery_bps)
        recovery_is_list.append(cumulative_recovery_is)

        Sw_avg_prev = Sw_avg_new_bps
        v_water_current = v_water_bps
        vel_water_list.append(v_water_current)

        print(f"Step {step_counter}: t = {t / hour:.2f} h, stage = '{current_stage}', "
              f"\u03b8_bps = {theta_bps:.3f}, \u03b8_is = {theta_is:.3f}, "
              f"rec_bps = {cumulative_recovery_bps:.2f}%, rec_is = {cumulative_recovery_is:.2f}%")

    print(" Time stepping completed =\n")

    time_arr = np.array(time_list)
    vel_water_arr = np.array(vel_water_list)
    recovery_bps_arr = np.array(recovery_bps_list)
    recovery_is_arr = np.array(recovery_is_list)

    results_dict = {}
    for name in species_results:
        results_dict[name] = {
            'time': time_arr,
            'c_t_in': np.array(species_results[name]['c_t_in']),
            'c_t_out': np.array(species_results[name]['c_t_out']),
        }

    global_eq_results = {}
    if run_eq:
        global_eq_results = {
            'time': time_arr,
            'bps_mid_time': np.array(bps_mid_list),
            'is_mid_time': np.array(is_mid_list),
        }

    results_dict['recovery_bps'] = {
        'time': time_arr,
        'recovery': recovery_bps_arr,
    }
    results_dict['recovery_is'] = {
        'time': time_arr,
        'recovery': recovery_is_arr,
    }

    return {
        'transport': results_dict,
        'global_eq': global_eq_results,
        'velocity_history': {
            'time': time_arr,
            'v_water': vel_water_arr,
        },
        'recovery_bps': {
            'time': time_arr,
            'recovery': recovery_bps_arr,
        },
        'recovery_is': {
            'time': time_arr,
            'recovery': recovery_is_arr,
        }
    }


### RESULT PLOTTING FUNCTIONS


In [23]:

def plot_results_multi(results_dict, params_list):
    print(f"\n{'=' * 60}")
    print("Plotting Transport Results")
    print(f"{'=' * 60}")
    for params in params_list:
        name = params['name']
        if name not in results_dict:
            continue
        res = results_dict[name]
        time_h = res['time'] / hour
        XYplot(time_h[1:], res['c_t_in'][1:], 0.5, t_f / hour,
               'Time [hours]', f'C_{name} inlet [mol]', 'r-',
               fname_stem=f'timeseries_{name}_inlet', subfolder='Transport_Figures')
        XYplot(time_h[1:], res['c_t_out'][1:], 0.5, t_f / hour,
               'Time [hours]', f'C_{name} outlet [mol]', 'b-',
               fname_stem=f'timeseries_{name}_outlet', subfolder='Transport_Figures')
        pd.DataFrame(res['c_t_out']).to_csv(os.path.join(Transport_Path, f'c_t_out_{name}.csv'), index=False)

    species_colors = {
        'Na': '#1f77b4', 'Ca': '#d62728', 'Mg': '#2ca02c',
        'Cl': '#9467bd', 'SO4': '#ff7f0e', 'HCO3': '#FFA500',
    }
    fig, ax = plt.subplots(figsize=(10, 7))
    for params in params_list:
        name = params['name']
        if name not in results_dict:
            continue
        res = results_dict[name]
        time_h = res['time'] / hour
        ax.plot(time_h[1:], res['c_t_out'][1:], color=species_colors.get(name, 'k'),
                linewidth=2, label=name)
    ax.set_xlim(0.5, t_f / hour)
    ax.set_xlabel('Time [hours]', fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    ax.set_ylabel('Outlet Concentration [mol]', fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    ax.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    ax.legend(fontsize=LEGEND_FONTSIZE, frameon=True)
    ax.grid(False)
    plt.tight_layout()
    save_fig(fig, 'timeseries_outlet_all_species', subfolder='Transport_Figures')
    plt.show(block=False)
    plt.pause(0.1)


def plot_velocity_history(velocity_history):
    time_h = velocity_history['time'] / hour
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(time_h[1:], velocity_history['v_water'][1:], 'b-', linewidth=2, label='Water (coupled BL)')
    ax.set_xlim(0.5, t_f / hour)
    ax.set_xlabel('Time [hours]', fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    ax.set_ylabel('BL Velocity [m/s]', fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_WEIGHT)
    ax.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    ax.set_title('Coupled Buckley-Leverett Velocity Used in Transport', fontsize=TITLE_FONTSIZE, fontweight='bold')
    ax.legend(fontsize=LEGEND_FONTSIZE, frameon=True)
    ax.grid(False)
    plt.tight_layout()
    save_fig(fig, 'BL_velocity_coupled', subfolder='Transport_Figures')
    plt.show(block=False)
    plt.pause(0.1)
    pd.DataFrame(velocity_history).to_csv(os.path.join(Transport_Path, 'coupled_velocity_history.csv'), index=False)


def plot_recovery(recovery_bps, recovery_is, exp_data=None):
    time_h_bps = recovery_bps['time'] / hour
    rec_bps = recovery_bps['recovery']
    time_h_is = recovery_is['time'] / hour
    rec_is = recovery_is['recovery']

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(time_h_bps, rec_bps, 'k-', linewidth=2.5, label='Bond Product Sum')
    ax.plot(time_h_is, rec_is, 'r--', linewidth=2.0, label='Ionic Strength')

    if exp_data is not None:
        ax.plot(exp_data[:, 0], exp_data[:, 1], 'yo', markersize=6, label='Experimental data', markeredgecolor='black')

    t1 = CONCENTRATION_STAGES['HSWF']['t_end'] / hour
    t2 = CONCENTRATION_STAGES['MSWF']['t_end'] / hour
    ax.axvline(t1, color='gray', linestyle=':', alpha=0.7, label=None)
    ax.axvline(t2, color='gray', linestyle=':', alpha=0.7, label=None)

    stage_times = [
        (0.0, t1, 'HSWF'),
        (t1, t2, 'MSWF'),
        (t2, CONCENTRATION_STAGES['LSWF']['t_end'] / hour, 'LSWF')
    ]
    y_pos = 0.0
    for start, end, label in stage_times:
        mid = (start + end) / 2
        ax.text(mid, y_pos, label, ha='center', va='bottom',
                fontsize=12, fontweight='bold', color='black')

    ax.set_xlabel('Time [hours]', fontsize=14, fontweight='bold')
    ax.set_ylabel('Oil recovery [% OIIP]', fontsize=14, fontweight='bold')
    ax.set_xlim(0, CONCENTRATION_STAGES['LSWF']['t_end'] / hour)

    ymax_plot = max(np.max(rec_bps), np.max(rec_is),
                     np.max(exp_data[:, 1]) if exp_data is not None else 0)
    ax.set_ylim(0, ymax_plot * 1.1 if ymax_plot > 0 else 100)

    ax.legend(loc='best')
    ax.grid(False)
    plt.tight_layout()
    save_fig(fig, 'Oil_recovery_comparison_BPS_IS', subfolder='Figures')
    plt.show(block=False)
    plt.pause(0.1)


# SIMULATION ORCHESTRATION

In [24]:

def run_simulation(run_mode='transport+equilibrium', use_equilibrium=True):
    run_trans, run_eq = resolve_modes(run_mode, use_equilibrium)

    mesh_dict = create_mesh_and_domain()
    space_dict = create_function_spaces(mesh_dict, num_components=6)
    boundary_dict = setup_boundary_parts(mesh_dict)
    params_list = setup_physical_parameters_multi(STAGE_ORDER[0])
    transport_bc_dict = setup_transport_boundary_conditions(space_dict, params_list, boundary_dict)
    init_dict = setup_initial_conditions_multi(space_dict, params_list)

    results = solve_transport_problem(
        space_dict, boundary_dict, transport_bc_dict,
        init_dict, params_list, run_trans, run_eq)

    plot_results_multi(results['transport'], params_list)
    plot_velocity_history(results['velocity_history'])
    plot_recovery(results['recovery_bps'], results['recovery_is'], exp_data=data_exp)

    if run_eq and results['global_eq']:
        plot_wettability_altered_kr(results['global_eq'])

    print("\n=== Coupled simulation finished successfully ===")
    return results


# MAIN EXECUTION

In [1]:

#if __name__ == '__main__':
#    _t_start = time.time()
#    run_simulation()
#    _t_elapsed = time.time() - _t_start
#    print(f"\n=== Total runtime: {_t_elapsed:.2f} seconds "
#          f"({_t_elapsed/60:.2f} minutes, {_t_elapsed/3600:.3f} hours) ===")